# Paired Prompt Patching: Clean vs Corrupt

This 50-minute-track notebook is intentionally self-contained: it uses a seeded tiny PyTorch model, no downloads, and only public TorchLens APIs. The goal is to show the full workflow shape you would use on a larger model while keeping every cell runnable on CPU.


Paired prompt patching compares a clean input and a corrupted input, then asks which clean internal activation repairs the corrupted output. This is the smaller sibling of a full causal trace and is often the first debugging pass before a broad sweep.


In [ ]:
from __future__ import annotations

import math
from typing import Any

import matplotlib.pyplot as plt
import torch
from torch import nn

import torchlens as tl

SEED = 1606
torch.manual_seed(SEED)
torch.set_grad_enabled(False)


class TinyBlock(nn.Module):
    """One residual MLP block used as a transformer-sized stand-in."""

    def __init__(self, width: int) -> None:
        """Initialize the block."""
        super().__init__()
        self.ln = nn.LayerNorm(width)
        self.up = nn.Linear(width, width)
        self.down = nn.Linear(width, width)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply layer norm, MLP, and residual addition."""
        hidden = torch.relu(self.up(self.ln(x)))
        return x + self.down(hidden)


class TinyTransformer(nn.Module):
    """Small token model with named blocks and sequence positions."""

    def __init__(self, vocab_size: int = 16, width: int = 12, depth: int = 3) -> None:
        """Initialize embeddings, blocks, and unembedding head."""
        super().__init__()
        self.embed = nn.Embedding(vocab_size, width)
        self.blocks = nn.ModuleList([TinyBlock(width) for _ in range(depth)])
        self.unembed = nn.Linear(width, vocab_size, bias=False)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """Return logits for every token position."""
        x = self.embed(tokens)
        for block in self.blocks:
            x = block(x)
        return self.unembed(x)

In [ ]:
model = TinyTransformer(vocab_size=14, width=10, depth=2).eval()
clean_tokens = torch.tensor([[2, 3, 4, 5]])
corrupt_tokens = torch.tensor([[2, 8, 4, 5]])
clean = tl.log_forward_pass(
    model, clean_tokens, vis_opt="none", intervention_ready=True, name="clean"
)
corrupt = tl.log_forward_pass(
    model, corrupt_tokens, vis_opt="none", intervention_ready=True, name="corrupt"
)
site = clean.find_sites(tl.func("relu")).first()
print(site.layer_label)

In [ ]:
patched = corrupt.fork("patched_with_clean_relu")
patched.set(tl.label(site.layer_label), site.activation, confirm_mutation=True)
patched.replay()

clean_out = clean.layer_list[-1].activation
corrupt_out = corrupt.layer_list[-1].activation
patched_out = patched.layer_list[-1].activation
clean_distance = torch.linalg.vector_norm(clean_out - corrupt_out)
patched_distance = torch.linalg.vector_norm(clean_out - patched_out)
print(f"corrupt distance from clean: {clean_distance:.4f}")
print(f"patched distance from clean: {patched_distance:.4f}")

In [ ]:
bundle = tl.bundle({"clean": clean, "corrupt": corrupt, "patched": patched}, baseline="clean")
last_site_comparison = bundle.metric(
    lambda member: float(torch.linalg.vector_norm(member.layer_list[-1].activation - clean_out))
)
print(last_site_comparison)